# Preprocessing and leakage prevention

**Research objective.** Demonstrate the exact saved text and image preprocessing boundaries.

All results are read from local FNEWS artifacts. `SOURCE_TRUE` and `SOURCE_NON_TRUE` are Fakeddit source-taxonomy labels, not independently verified factuality labels. The held-out test partition is not opened by these notebooks except where a prior frozen metric is explicitly quoted.

In [1]:
from pathlib import Path
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image

warnings.filterwarnings("ignore", category=FutureWarning)
ROOT = Path.cwd()
if not (ROOT / "data/research_dataset").exists():
    ROOT = Path("/Users/aka_ank/Desktop/FNEWS")
DATA = ROOT / "data/research_dataset"
REPORTS = ROOT / "reports"
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.titlesize": 13, "axes.labelsize": 11})

def paired(split):
    frame = pd.read_parquet(DATA / "manifests" / f"{split}_manifest.parquet")
    return frame.loc[(frame.source_dataset == "Fakeddit") & (frame.download_status == "verified_reused_or_processed")].copy().reset_index(drop=True)

def canonical_source_labels(raw):
    """Fakeddit raw 1=true; return 0=SOURCE_TRUE, 1=SOURCE_NON_TRUE."""
    return 1 - np.asarray(raw, dtype=int)

def source_names(values):
    return np.where(np.asarray(values) == 0, "SOURCE_TRUE", "SOURCE_NON_TRUE")

def show_frame(frame, rows=10):
    display(frame.head(rows))


## Train-only preprocessing principle

Vocabulary, inverse-document-frequency statistics, calibration parameters, and model selection must be fitted on training data only. Validation is used for recorded selection/audit tasks; the frozen test result is never reused here.

In [2]:
train,val=paired('train'),paired('validate');model=joblib.load(REPORTS/'experiments/research_dataset_baselines/tfidf_svm/model.joblib');tfidf=model.named_steps['tfidf'];svm=model.named_steps['svm'];sample=train.iloc[0];valid=val.iloc[0]
print('Train sample:',sample.text);print('Validation sample:',valid.text);print('SVM classes:',svm.classes_.tolist());print('TF-IDF configuration:',tfidf.get_params())
print('Raw Fakeddit 1 = SOURCE_TRUE; raw 0 = SOURCE_NON_TRUE')

Train sample: i come in peace
Validation sample: found him hiding behind my monitor what a joker
SVM classes: [0, 1]
TF-IDF configuration: {'analyzer': 'word', 'binary': False, 'decode_error': 'strict', 'dtype': <class 'numpy.float64'>, 'encoding': 'utf-8', 'input': 'content', 'lowercase': True, 'max_df': 1.0, 'max_features': 50000, 'min_df': 2, 'ngram_range': (1, 2), 'norm': 'l2', 'preprocessor': None, 'smooth_idf': True, 'stop_words': None, 'strip_accents': None, 'sublinear_tf': True, 'token_pattern': '(?u)\\b\\w\\w+\\b', 'tokenizer': None, 'use_idf': True, 'vocabulary': None}
Raw Fakeddit 1 = SOURCE_TRUE; raw 0 = SOURCE_NON_TRUE


In [3]:
x_train=tfidf.transform([sample.text]);x_val=tfidf.transform([valid.text]);print('Train transform shape:',x_train.shape,'nonzero:',x_train.nnz);print('Validation transform shape:',x_val.shape,'nonzero:',x_val.nnz);print('No refit is performed on validation.')

Train transform shape: (1, 2964) nonzero: 4
Validation transform shape: (1, 2964) nonzero: 6
No refit is performed on validation.


## Text processing

The saved vectorizer lowercases and produces unigram/bigram sparse vectors. Stopword removal is deliberately not introduced because short titles can lose useful grammatical context. Whitespace cleanup occurs upstream in `clean_title`.

In [4]:
from torchvision.models import ResNet18_Weights
transform=ResNet18_Weights.DEFAULT.transforms();image=Image.open(DATA/valid.image_path).convert('RGB');tensor=transform(image);print('Original image:',image.size,'RGB');print('ResNet input tensor:',tuple(tensor.shape),tensor.dtype)
print('Image path:',valid.image_path)

Original image: (320, 426) RGB
ResNet input tensor: (3, 224, 224) torch.float32
Image path: images/fakeddit/validate/11nlcg.jpg


## Leakage controls

Official split membership is preserved. Exact hashes and perceptual hashes were used to exclude cross-split duplicate or near-duplicate images. The application validates uploaded image bytes before persistence; URL extraction applies SSRF, redirect, robots, content-type, and size controls.